# xBD — Exploratory Data Analysis & Data Cleaning

**Author:** Artur Zavistovskyi · **Team part:** EDA + data cleaning
**Project:** Mapping Natural-Disaster Damage from Satellite Imagery
**Source:** sections 1–4 of `artur_xbd_NESH_CAN_RUN.ipynb`, split out and rewritten for a
**local, Kaggle-free** run against the xView2 archives extracted under `data/`.

The companion notebook [`02_data_split.ipynb`](02_data_split.ipynb) carries section 5 (the split
audit and the scene sampling). Model training, patch extraction and evaluation stay in the
model notebooks — nothing in here needs torch or a GPU.

---

## What this notebook does

`splits.csv → label scan → EDA → cleaning rules and quality diagnostics → cleaned polygon table`

Each analysis is followed by **what we see → why it matters → what it changes downstream**.

## Running it

```
pip install numpy pandas matplotlib pillow pyarrow notebook
```

Start Jupyter from the project root, or set `PROJECT_ROOT_OVERRIDE` in the PATHS cell.
Expected layout — the xView2 tars extracted under `data/`:

```
<project root>/
    data/
        splits.csv                <- the team split index (scripts/prepare_splits.py)
        train/{images,labels}     <- train_images_labels_targets.tar   == xBD "tier1"
        test/{images,labels}      <- test_images_labels_targets.tar    == xBD "test"
        tier3/{images,labels}     <- tier3.tar          (all wildfire events except socal/santa-rosa)
        hold/{images,labels}      <- hold_images_labels_targets.tar
```

The `xbd/{hold,test,tier1,tier3}/` layout from `docs/local_setup.md` is accepted too — the
resolver tries both. **A partial dataset is also fine**: the coverage cell drops scenes whose
files are absent, prints exactly what is missing, and keys every cache on the coverage so a
partial scan is never silently reused once the rest of the data arrives.

**Outputs** — `work/cache/` (parquet scans, so a re-run skips the slow steps) and
`results/01_eda/` (figures + CSV tables). All derived data; gitignored.

## Two deliberate design choices

- **Paths are rebuilt from `(origin, subdir, name)`**, never from a directory scan. Some xBD
  mirrors ship an extra `train/` duplicate of `tier1`; globbing would place the same scene in
  two splits.
- **Nothing is deleted silently.** Every cleaning rule states its threshold, its cost, and its
  per-split removal rate; anything that cannot be justified is flagged for review instead.

## 1 · Setup

In [ ]:
# ---------------------------------------------------------------- dependency check
# Neither of these two notebooks needs torch: EDA and the split are pure pandas/PIL work.
# Fail here with an actionable message rather than with an ImportError halfway down.
import importlib.util

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
            "PIL": "pillow", "pyarrow": "pyarrow"}          # pyarrow: the .parquet caches

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) + "\n\n"
                      "    pip install " + " ".join(missing))
print("all required packages present")

In [ ]:
# ---------------------------------------------------------------- imports
import os, sys, json, re, time, random, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

warnings.filterwarnings("ignore", category=UserWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = False   # we WANT truncated files to raise, so we can count them
Image.MAX_IMAGE_PIXELS = None

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ================================================================ CONFIGURATION
# Everything tunable lives here. Nothing below this cell hard-codes a magic number.

# --- labels: the xBD 4-level joint damage scale -------------------------------
DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
NAME_TO_IDX    = {n: i for i, n in enumerate(DAMAGE_CLASSES)}
NUM_CLASSES    = len(DAMAGE_CLASSES)
SHORT          = ["none", "minor", "major", "destr"]      # compact axis labels

# "un-classified" is a 5th value annotators used when a building could NOT be assessed.
# It is not a damage level, so it is excluded from training - but it is our single best
# occlusion signal, so we keep and analyse it in section 4.3.
UNCLASSIFIED   = "un-classified"

# --- the four xBD origin directories (xbd/train/ is a duplicate of tier1, excluded) ---
ORIGINS = ("hold", "test", "tier1", "tier3")
SPLITS  = ("train", "val", "test_id", "test_ood")

# The five held-out wildfire events, frozen here so plots stay correct even when part of
# the dataset is not on disk yet (see scripts/prepare_splits.py on Bhuvanesh-branch).
FIRE_EVENTS = {"socal-fire", "santa-rosa-wildfire",
               "woolsey-fire", "pinery-bushfire", "portugal-wildfire"}

# --- patch geometry (set by EDA 3.4, consumed by the model notebooks) ---------
PATCH_SIZE   = 64     # px. Justified against the measured bbox distribution in section 3.4.
BBOX_PAD     = 10     # px of context kept around each building footprint
MIN_BBOX_PX  = 8      # cleaning rule: drop footprints smaller than this (section 4.5)
MIN_BUILDINGS_PER_SCENE = 5    # scene-sampling filter, calibrated in section 5.2

# Scene budget per split - how many scenes the model notebooks are allowed to extract from.
SCENE_BUDGET = {"train": 800, "val": 200, "test_id": 300, "test_ood": 400}

# --- diagnostics --------------------------------------------------------------
QC_SAMPLE_SCENES = 400    # scenes used for pixel-level image QC and duplicate hashing
OCCLUSION_TOP_N  = 12     # occlusion candidates displayed for manual review
SHOW_EXAMPLE_GRIDS = True # section 3.7 galleries; set False if the disk is slow (cosmetic only)

# --- I/O ----------------------------------------------------------------------
# PIL releases the GIL while decoding, so threads help even for a pure-Python loop.
# On a spinning disk, lower this: concurrent random reads make seek time worse.
IO_WORKERS = 16

# --- reproducibility ----------------------------------------------------------
SEED = 42            # same seed as prepare_splits.py, so anything re-derived here matches

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed()
print("patch", PATCH_SIZE, "| seed", SEED, "| scene budget", SCENE_BUDGET)

In [ ]:
# ================================================================ PATHS (local run)
# The imagery layout in use here - the Kaggle xbd-dataset archive extracted under data_same/:
#
#   <project root>/
#       data/
#           splits.csv                     <- the team split (scripts/prepare_splits.py)
#       data_same/
#           xbd/{hold,test,tier1,tier3}/{images,labels,masks}
#
# Two other layouts are accepted without editing anything, because the resolver below searches
# several roots: xbd/ directly in the project root (docs/local_setup.md), and the per-archive
# form data/{train,test,tier3,hold}/ where `train` is the xView2 tar that IS xBD "tier1".

PROJECT_ROOT_OVERRIDE = None    # e.g. r"D:\UNI\DLSS\satellite-disaster-damage-mapping"


def find_project_root():
    """Look for data/splits.csv - in the override, then cwd, then its parents."""
    cands = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else []
    here = Path.cwd().resolve()
    cands += [here, *here.parents]
    for d in cands:
        if (d / "data" / "splits.csv").is_file():
            return d
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/splits.csv not found.\n"
        "Expected a directory containing data/splits.csv (the team split index).\n"
        "Either start Jupyter from the project root, or set PROJECT_ROOT_OVERRIDE above.\n"
        f"Searched upwards from: {Path.cwd()}")

DATA       = PROJECT_ROOT / "data"          # the split index lives here, not the imagery
SPLITS_CSV = DATA / "splits.csv"

# Roots searched for the imagery, in priority order. data_same/ holds the complete Kaggle
# archive and wins over anything left in data/ from an earlier partial download.
IMAGE_ROOTS = [PROJECT_ROOT / "data_same", DATA, PROJECT_ROOT]

WORK    = PROJECT_ROOT / "work";           WORK.mkdir(exist_ok=True)
CACHE   = WORK / "cache";                  CACHE.mkdir(parents=True, exist_ok=True)
RESULTS = PROJECT_ROOT / "results" / "01_eda"; RESULTS.mkdir(parents=True, exist_ok=True)

print("cwd          :", Path.cwd())
print("project root :", PROJECT_ROOT)
print("splits.csv   :", SPLITS_CSV, f"({SPLITS_CSV.stat().st_size / 1e6:.1f} MB)")
print("cache        :", CACHE)
print("figures/CSVs :", RESULTS)

In [ ]:
# ---------------------------------------------------------------- resolve the origin directories
# splits.csv records paths as xbd\<origin>\<subdir>\<name>. We never use that string directly:
# we rebuild the path from (origin, subdir, name) so ANY on-disk layout works as long as each
# origin directory can be located.
#
# Two ordering rules matter here:
#   - roots are the OUTER loop, so a complete data_same/ wins over leftovers in data/;
#   - "xbd/tier1" is tried before "xbd/train", because the Kaggle mirror ships a partial
#     xbd/train/ that duplicates tier1 and is absent from splits.csv. Requiring BOTH images/
#     and labels/ rejects it a second time - it has no labels/ - but order alone is enough.

ORIGIN_DIR_CANDIDATES = {
    "tier1": ["xbd/tier1", "tier1", "train", "xbd/train"],   # the xView2 "train" tar IS tier1
    "test":  ["xbd/test",  "test"],
    "tier3": ["xbd/tier3", "tier3"],
    "hold":  ["xbd/hold",  "hold"],
}

ORIGIN_ROOT = {}
for origin, cands in ORIGIN_DIR_CANDIDATES.items():
    for base in IMAGE_ROOTS:
        for c in cands:
            p = base / c
            if (p / "images").is_dir() and (p / "labels").is_dir():
                ORIGIN_ROOT[origin] = p
                break
        if origin in ORIGIN_ROOT:
            break

print("origin directories found on disk:")
for origin in ORIGINS:
    root = ORIGIN_ROOT.get(origin)
    if root is None:
        print(f"  {origin:6s} -> MISSING")
    else:
        n_img = sum(1 for _ in (root / "images").glob("*.png"))
        print(f"  {origin:6s} -> {root}   ({n_img:,} image files)")

if not ORIGIN_ROOT:
    raise FileNotFoundError(
        "No xBD origin directory found. Searched these roots:\n"
        + "".join(f"  {r}\n" for r in IMAGE_ROOTS) +
        "Extract the xView2 imagery into one of them, as either xbd/{hold,test,tier1,tier3}/\n"
        "or {hold,test,tier3,train}/ - each with images/ and labels/ inside.")

## 2 · Dataset inspection

### 2.1 Load the team split

`splits.csv` is the output of `scripts/prepare_splits.py` (Bhuvanesh). One row per file, 7 files
per scene: `images/{pre,post}`, `labels/{pre,post}`, `masks/{pre,post,post_rgb}`.

We keep only `images` and `labels` — the `masks/` folder holds **segmentation** targets, which
this per-building **classification** task does not use. (The local xView2 tars name that folder
`targets/`; same content, equally unused.)

| split | rule | what it measures |
|---|---|---|
| `train` / `val` | tier1+tier3, non-fire, 88.9 / 11.1 stratified by event | fitting and model selection |
| `test_id` | official `test` + `hold` directories, non-fire | in-distribution reference |
| `test_ood` | every wildfire scene, any origin | zero-shot, the headline result |

In [ ]:
# ---------------------------------------------------------------- splits.csv -> one row per scene
splits_raw = pd.read_csv(SPLITS_CSV)
print("splits.csv:", splits_raw.shape)
display(splits_raw.head(3))

# Keep images + labels, drop masks: masks/ holds SEGMENTATION targets, which this per-building
# CLASSIFICATION task does not use. (The local tars call that directory `targets` - same thing,
# and equally unused, which is why the name mismatch does not matter.)
keep = splits_raw[splits_raw.subdir.isin(["images", "labels"])].copy()
keep["abs_path"] = [
    str(ORIGIN_ROOT[o] / s / n) if o in ORIGIN_ROOT else ""
    for o, s, n in zip(keep.origin, keep.subdir, keep["name"])
]
keep["col"] = keep.subdir.map({"images": "img", "labels": "lab"}) + "_" + keep.kind

scenes = (keep.pivot_table(index=["scene_id", "disaster", "origin", "split"],
                           columns="col", values="abs_path", aggfunc="first")
              .reset_index())
scenes.columns.name = None
scenes = scenes.rename(columns={"img_pre_disaster": "img_pre", "img_post_disaster": "img_post",
                                "lab_pre_disaster": "lab_pre", "lab_post_disaster": "lab_post"})
scenes["is_fire"] = scenes.disaster.isin(FIRE_EVENTS)

assert scenes.scene_id.is_unique, "scene_id must be unique after the pivot"
assert set(scenes.loc[scenes.split == "test_ood", "disaster"]) <= FIRE_EVENTS, \
    "test_ood must contain wildfire events only"

print(f"\nsplits.csv describes {len(scenes):,} scenes across {scenes.disaster.nunique()} events")
display(pd.crosstab(scenes.origin, scenes.split).reindex(index=ORIGINS, columns=SPLITS,
                                                         fill_value=0))

### 2.2 Coverage — reconcile `splits.csv` against this machine

`splits.csv` describes the full dataset that `prepare_splits.py` indexed. This is the one cell
that decides what the rest of the notebook actually runs on.

In [ ]:
# ---------------------------------------------------------------- what is ACTUALLY on disk
# splits.csv describes the full 11,034-scene dataset. This local machine may only hold part of
# it. A scene is usable only if pre image + post image + post label all exist; anything else is
# dropped here, once, loudly - rather than failing 200 cells later inside a thread pool.

NEEDED = ["img_pre", "img_post", "lab_post"]

t0 = time.time()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    present = {c: np.fromiter(ex.map(os.path.exists, scenes[c]), bool, len(scenes))
               for c in NEEDED}
scenes["files_ok"] = np.logical_and.reduce(list(present.values()))
print(f"checked {3 * len(scenes):,} paths in {time.time() - t0:.1f}s\n")

cov = (scenes.groupby(["origin", "split"]).files_ok.agg(on_disk="sum", in_splits_csv="size")
             .reset_index())
cov["coverage_%"] = (100 * cov.on_disk / cov.in_splits_csv).round(1)
print("scene coverage, origin x split:")
display(cov)

by_split = scenes.groupby("split").files_ok.agg(on_disk="sum", in_splits_csv="size").reindex(SPLITS)
by_split["coverage_%"] = (100 * by_split.on_disk / by_split.in_splits_csv).round(1)
print("scene coverage per split:")
display(by_split)

FULL_DATASET = bool(scenes.files_ok.all())
scenes_all, scenes = scenes, scenes[scenes.files_ok].reset_index(drop=True)

if FULL_DATASET:
    print("\nComplete dataset on disk - results are directly comparable to the reference run.")
else:
    lost = [o for o in ORIGINS if o not in ORIGIN_ROOT
            or not scenes_all.loc[scenes_all.origin == o, "files_ok"].any()]
    print(f"\n!! PARTIAL DATASET: {len(scenes):,} of {len(scenes_all):,} scenes "
          f"({100 * len(scenes) / len(scenes_all):.1f}%) are on disk.")
    print(f"   Missing or empty origins: {lost}")
    print("   Every number below is computed on what IS present and is therefore NOT")
    print("   comparable to the reference full-dataset run. Add the missing archives and")
    print("   re-run to reproduce it - the cache is keyed on coverage, so nothing is stale.")

if scenes.empty:
    raise RuntimeError("No usable scenes on disk. Extract the xView2 archives first.")

# Cache files are keyed on coverage: adding tier3/ or hold/ later changes the key, so a partial
# scan can never be silently reused as if it were the full one.
COVERAGE_TAG = "-".join(sorted(ORIGIN_ROOT)) + f"_{len(scenes)}"
print("\ncoverage tag (cache key):", COVERAGE_TAG)

fire_events = sorted(set(scenes.disaster) & FIRE_EVENTS)
print("wildfire events present   :", fire_events)
leak = scenes[scenes.split.isin(["train", "val"]) & scenes.disaster.isin(FIRE_EVENTS)]
assert len(leak) == 0, "wildfire leaked into the training pool"
print("fire scenes in train/val  :", len(leak), "(must be 0)")

**On the reference (complete) dataset.** 11,034 scenes across 19 events. Five wildfire events
(6,372 scenes) sit entirely in `test_ood`; `train`+`val` hold 3,527 non-fire scenes. No wildfire
leaks into training.

**Why it matters.** `test_ood` is larger than the whole training pool, so any dataset-wide
statistic is dominated by wildfire. Every distribution below is therefore reported **per split**.

**Downstream.** Scene sampling is stratified by event, and class weights and thresholds are
computed on `train` only.

### 2.3 Read the labels

Damage labels live in the **post-disaster JSON**, not in `splits.csv`. Structure per file:

```
features.xy[]      -> {"properties": {"subtype": ..., "uid": ...}, "wkt": "POLYGON ((x y, ...))"}
features.lng_lat[] -> the same polygons in WGS84
metadata           -> capture geometry: gsd, off_nadir_angle, sun_elevation, capture_date, ...
```

Two things worth flagging for the team:

- Pre-disaster JSONs carry footprints with **no** `subtype` — they are localisation only.
- `features.lng_lat` gives real lat/lon per building, so **Part 3 does not need the
  `xview_geotransforms.json` file** that the README lists.

We print the real keys of one file rather than assuming them.

In [ ]:
probe = json.loads(Path(scenes.lab_post.iloc[0]).read_text())
print("top-level keys :", list(probe.keys()))
print("features keys  :", list(probe["features"].keys()))
print("metadata keys  :", list(probe["metadata"].keys()))
print("\none xy feature :")
print(json.dumps(probe["features"]["xy"][0], indent=2)[:600])

In [ ]:
_WKT_NUM = re.compile(r"-?\d+\.?\d*")


def wkt_bbox(wkt: str):
    """Axis-aligned pixel bbox of a WKT polygon. Numbers alternate x, y."""
    nums = [float(v) for v in _WKT_NUM.findall(wkt)]
    xs, ys = nums[0::2], nums[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def read_post_label(row):
    """Parse one post-disaster JSON -> (per-building records, per-scene metadata record)."""
    try:
        d = json.loads(Path(row.lab_post).read_text())
    except Exception as e:                    # counted as a corrupted label in section 4.2
        return [], {"scene_id": row.scene_id, "label_error": type(e).__name__,
                    "n_buildings": 0, "n_unclassified": 0}

    meta, feats = d.get("metadata", {}), d["features"].get("xy", [])
    recs, n_unc = [], 0
    for f in feats:
        props = f.get("properties", {})
        sub = props.get("subtype", UNCLASSIFIED)
        n_unc += (sub == UNCLASSIFIED)
        minx, miny, maxx, maxy = wkt_bbox(f["wkt"])
        recs.append((row.scene_id, props.get("uid", ""), sub, minx, miny, maxx, maxy))

    scene_rec = {
        "scene_id": row.scene_id, "label_error": None,
        "n_buildings": len(feats), "n_unclassified": n_unc,
        "gsd": meta.get("gsd"), "off_nadir_angle": meta.get("off_nadir_angle"),
        "sun_elevation": meta.get("sun_elevation"), "capture_date": meta.get("capture_date"),
        "disaster_type": meta.get("disaster_type"),
        "img_width": meta.get("width"), "img_height": meta.get("height"),
    }
    return recs, scene_rec

In [ ]:
# ---------------------------------------------------------------- full label scan (cached)
BLD_CACHE = CACHE / f"buildings__{COVERAGE_TAG}.parquet"
SCN_CACHE = CACHE / f"scene_meta__{COVERAGE_TAG}.parquet"

if BLD_CACHE.exists() and SCN_CACHE.exists():
    buildings  = pd.read_parquet(BLD_CACHE)
    scene_meta = pd.read_parquet(SCN_CACHE)
    print("loaded label scan from cache:", BLD_CACHE.name)
else:
    t0 = time.time()
    rows = list(scenes.itertuples(index=False))
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:      # I/O bound: many small JSON reads
        out = list(ex.map(read_post_label, rows))
    buildings = pd.DataFrame([r for recs, _ in out for r in recs],
                             columns=["scene_id", "uid", "subtype",
                                      "minx", "miny", "maxx", "maxy"])
    scene_meta = pd.DataFrame([s for _, s in out])
    buildings.to_parquet(BLD_CACHE, index=False)
    scene_meta.to_parquet(SCN_CACHE, index=False)
    print(f"scanned {len(rows):,} label files in {time.time() - t0:.0f}s")

# attach split / disaster / origin to every row and derive bbox geometry
buildings = buildings.merge(scenes[["scene_id", "disaster", "origin", "split"]],
                            on="scene_id", how="left")
buildings["bw"]     = buildings.maxx - buildings.minx
buildings["bh"]     = buildings.maxy - buildings.miny
buildings["barea"]  = buildings.bw * buildings.bh
buildings["aspect"] = buildings.bw / buildings.bh.clip(lower=1e-6)
buildings["label"]  = buildings.subtype.map(NAME_TO_IDX)     # NaN for un-classified

scene_meta = scene_meta.merge(scenes[["scene_id", "disaster", "origin", "split"]],
                              on="scene_id", how="left")

print(f"\n{len(buildings):,} building polygons across "
      f"{buildings.scene_id.nunique():,} post-disaster scenes")
display(buildings.head(3))

In [ ]:
# ---------------------------------------------------------------- raw label inventory
print("subtype values found (the ground-truth vocabulary, measured not assumed):")
vc = buildings.subtype.value_counts(dropna=False)
display(vc.rename("polygons").to_frame().assign(pct=(100 * vc / len(buildings)).round(2)))

unexpected = set(vc.index) - set(DAMAGE_CLASSES) - {UNCLASSIFIED}
print("unexpected subtype values :", unexpected if unexpected else "none")
print("scenes with an unreadable label file :", int(scene_meta.label_error.notna().sum()))
print("scenes with ZERO building polygons   :", int((scene_meta.n_buildings == 0).sum()))

## 3 · Exploratory Data Analysis

### 3.1 Class distribution, per split

In [ ]:
def class_table(df, by):
    """Counts and row-normalised percentages of damage classes, grouped by `by`."""
    ct = pd.crosstab(df[by], df.subtype).reindex(columns=DAMAGE_CLASSES + [UNCLASSIFIED],
                                                 fill_value=0)
    pct = ct.div(ct.sum(axis=1).clip(lower=1), axis=0) * 100
    return ct, pct


ct_split, pct_split = class_table(buildings, "split")
ct_split  = ct_split.reindex(SPLITS, fill_value=0)
pct_split = pct_split.reindex(SPLITS, fill_value=0)

print("building polygons per split x damage class")
display(ct_split.assign(total=ct_split.sum(axis=1)))
print("row-normalised %")
display(pct_split.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ct_split[DAMAGE_CLASSES].plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Damage-class counts per split (log scale)")
axes[0].set_yscale("log"); axes[0].set_ylabel("polygons"); axes[0].set_xlabel("")
pct_split[DAMAGE_CLASSES].plot(kind="bar", stacked=True, ax=axes[1], legend=False)
axes[1].set_title("Damage-class composition per split (%)")
axes[1].set_ylabel("% of polygons"); axes[1].set_xlabel("")
plt.tight_layout()
fig.savefig(RESULTS / "damage_class_per_split.png", dpi=140, bbox_inches="tight")
plt.show()

ct_split.assign(total=ct_split.sum(axis=1)).to_csv(RESULTS / "damage_class_counts_all_splits.csv")
pct_split.round(2).to_csv(RESULTS / "damage_class_composition_per_split.csv")

imb = ct_split.loc["train", DAMAGE_CLASSES]
print(f"train imbalance ratio (majority / minority): {imb.max() / max(imb.min(), 1):.1f}x")

**What we see (reference run).** `no-damage` is 72.0% of training polygons and the rarest class is
11.6× smaller. The composition differs sharply by split — `test_ood` is 80.2 / 0.9 / 0.9 / 14.7 %
against 72.0 / 9.7 / 8.4 / 6.2 % in train.

**Why it matters.** Accuracy is close to useless here: always predicting `no-damage` scores 0.72 on
train while producing a damage map that never flags damage. **Macro-F1 and per-class recall decide
everything**, including model selection.

**Downstream.**
1. Inverse-frequency class weights from `train` only — they came out 0.33 / 2.38 / 2.91 / 4.92.
2. Checkpointing and early stopping key on **val macro-F1**, not accuracy.
3. Confusion matrices are reported raw *and* row-normalised, because raw counts hide the minority
   classes completely.

Weighting was chosen over resampling: oversampling would repeat the same few destroyed buildings
many times per epoch and invite memorisation, undersampling would discard most of the majority
signal. Weighting keeps every sample once and changes only its gradient contribution.

### 3.2 Class distribution per disaster event — the transfer question, quantified

In [ ]:
ct_dis, pct_dis = class_table(buildings, "disaster")
order = scenes.groupby("disaster").is_fire.first().sort_values().index.tolist()
pct_dis = pct_dis.reindex([d for d in order if d in pct_dis.index])

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pct_dis[DAMAGE_CLASSES + [UNCLASSIFIED]].values, cmap="magma", aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(SHORT + ["un-cls"])
ax.set_yticks(range(len(pct_dis))); ax.set_yticklabels(pct_dis.index, fontsize=8)
for i, ev in enumerate(pct_dis.index):
    if ev in fire_events:
        ax.get_yticklabels()[i].set_color("crimson")
        ax.get_yticklabels()[i].set_fontweight("bold")
    for j in range(5):
        v = pct_dis.iloc[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if v < 55 else "black")
ax.set_title("Damage-class composition per event (%)\nred = held-out wildfire", fontsize=10)
ax.grid(False); plt.colorbar(im, label="% of polygons"); plt.tight_layout()
fig.savefig(RESULTS / "damage_composition_per_event_heatmap.png", dpi=140, bbox_inches="tight")
plt.show()

fire_mix    = pct_dis.loc[pct_dis.index.isin(fire_events),  DAMAGE_CLASSES].mean()
nonfire_mix = pct_dis.loc[~pct_dis.index.isin(fire_events), DAMAGE_CLASSES].mean()
mix = pd.DataFrame({"non-fire events (train domain)": nonfire_mix.round(1),
                    "wildfire events (OOD domain)": fire_mix.round(1)})
display(mix)
mix.to_csv(RESULTS / "damage_composition_fire_vs_nonfire.csv")
pct_dis.round(2).to_csv(RESULTS / "damage_composition_per_event.csv")

**What we see (reference run).** The class mix is imbalanced *differently in every event*, and the
fire rows are qualitatively different from the rest:

| | no-damage | minor | major | destroyed |
|---|---|---|---|---|
| non-fire events | 70.2 | 9.9 | 7.2 | 6.5 |
| wildfire events | 78.8 | **1.2** | **1.1** | **14.8** |

Intermediate damage nearly vanishes under fire while `destroyed` more than doubles. Individual
events vary wildly too — `mexico-earthquake` is 99% undamaged, `hurricane-matthew` is 51% minor.

**Why it matters.** Fire produces a bimodal outcome: a building is untouched or it is gone. A model
that learns the hurricane prior will be systematically miscalibrated on fire even if its visual
features transfer perfectly.

**Downstream.** This predicts *what kind* of failure to expect at evaluation time. A pure prior
shift produces errors with a consistent direction; failed features produce diffuse errors. That is
the difference between "recalibrate the thresholds" and "this approach does not deploy on an unseen
disaster".

### 3.3 Buildings per scene — and the tier3 empty-tile problem

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for sp in SPLITS:
    v = scene_meta.loc[scene_meta.split == sp, "n_buildings"]
    if len(v):
        axes[0].hist(v, bins=np.arange(0, 260, 5), histtype="step", lw=1.6,
                     label=f"{sp} (n={len(v)})")
axes[0].set(xlabel="building polygons per scene", ylabel="scenes",
            title="Buildings per scene, by split")
axes[0].legend(fontsize=8)

empty = (scene_meta.assign(empty=scene_meta.n_buildings.eq(0))
                   .groupby("disaster").empty.mean().mul(100).sort_values())
empty.plot(kind="barh", ax=axes[1],
           color=["crimson" if d in fire_events else "steelblue" for d in empty.index])
axes[1].set(xlabel="% of scenes with ZERO buildings", ylabel="",
            title="Empty scenes per event (red = wildfire)")
axes[1].tick_params(labelsize=8)
plt.tight_layout()
fig.savefig(RESULTS / "empty_scenes_per_event.png", dpi=140, bbox_inches="tight")
plt.show()

summary = (scene_meta.groupby("split")
           .agg(scenes=("scene_id", "size"),
                empty_scenes=("n_buildings", lambda s: int((s == 0).sum())),
                median_buildings=("n_buildings", "median"),
                total_buildings=("n_buildings", "sum"))
           .reindex(SPLITS))
summary["empty_%"] = (100 * summary.empty_scenes / summary.scenes).round(1)
display(summary)
summary.to_csv(RESULTS / "scene_summary.csv")

**What we see (reference run).** Empty scenes are common and extremely event-dependent:

| split | scenes | empty | median buildings |
|---|---|---|---|
| train | 3,135 | 9.8% | 29 |
| val | 392 | 11.0% | 31.5 |
| test_id | 1,135 | 5.4% | 26 |
| test_ood | 6,372 | **54.1%** | **0** |

The tier3 fire events top the chart (`pinery-bushfire` highest) — they cover large stretches of
forest and scrubland.

**Why it matters.** An empty scene yields zero samples but still costs a full 1024×1024 PNG decode.
Sampling wildfire scenes uniformly would have spent most of the OOD budget on empty forest.

**Downstream.** `02_data_split.ipynb` samples only scenes with at least `MIN_BUILDINGS_PER_SCENE`
polygons. This is a *sampling* decision, not a cleaning one — nothing is deleted, and the same rule
is applied to all four splits so it cannot tilt the ID→OOD comparison.

### 3.4 Building footprint geometry — this is what sets `PATCH_SIZE`

In [ ]:
bb = buildings[buildings.label.notna()]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(np.clip(bb.bw, 0, 200), bins=80, alpha=.6, label="width")
axes[0].hist(np.clip(bb.bh, 0, 200), bins=80, alpha=.6, label="height")
axes[0].axvline(PATCH_SIZE - 2 * BBOX_PAD, color="k", ls="--",
                label=f"patch usable = {PATCH_SIZE - 2 * BBOX_PAD}px")
axes[0].set(xlabel="px", ylabel="polygons", title="bbox side length"); axes[0].legend(fontsize=8)

axes[1].hist(np.clip(bb.aspect, 0, 5), bins=80, color="teal")
axes[1].set(xlabel="width / height", title="aspect ratio")

for c in range(NUM_CLASSES):
    v = bb.loc[bb.label == c, "barea"]
    if len(v):
        axes[2].hist(np.log10(v.clip(lower=1)), bins=60, histtype="step", lw=1.5, label=SHORT[c])
axes[2].set(xlabel="log10(bbox area, px^2)", title="footprint area by damage class")
axes[2].legend(fontsize=8)
plt.tight_layout()
fig.savefig(RESULTS / "training_footprint_geometry.png", dpi=140, bbox_inches="tight")
plt.show()

q = bb[["bw", "bh"]].max(axis=1).quantile([.5, .75, .9, .95, .99]).round(1)
print("percentiles of max(bbox width, height), px:")
display(q.rename("px").to_frame())
q.rename("px").to_frame().to_csv(RESULTS / "footprint_size_percentiles.csv")

tiny = (bb.bw < MIN_BBOX_PX) | (bb.bh < MIN_BBOX_PX)
print(f"footprints with a side < {MIN_BBOX_PX}px : {int(tiny.sum()):,} ({100 * tiny.mean():.2f}%)")

**What we see (reference run).** Footprints are small: median longest side 35 px, 90th percentile
64 px, with a thin tail out to 349 px at the 99th. Only 3.51% of footprints have a side under 8 px.

**Why it matters.**
- `PATCH_SIZE = 64` covers roughly 90% of buildings at close to native resolution. Larger mostly
  upsamples empty ground at quadratic cost; smaller destroys the roof texture that separates
  `minor` from `major`.
- The crop is resized to a square, so aspect ratio is not preserved — an accepted trade-off, and the
  measured distribution is close to square anyway.
- `BBOX_PAD = 10` px is what lets the model see the surroundings: debris, scorch marks, standing
  water. Damage is often more visible around a building than on it.

**Downstream.** Rule 4.5 drops the sub-8 px footprints — after padding and upscaling they are
interpolation artefacts, not observations of a building.

### 3.5 Capture metadata — the hidden domain shift

In [ ]:
meta_cols = ["gsd", "off_nadir_angle", "sun_elevation"]
geo = scene_meta.dropna(subset=meta_cols)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, c in zip(axes, meta_cols):
    for grp, lbl, col in [(geo[~geo.disaster.isin(fire_events)], "non-fire (train domain)", "steelblue"),
                          (geo[geo.disaster.isin(fire_events)],  "wildfire (OOD)", "crimson")]:
        if len(grp):
            ax.hist(grp[c], bins=40, alpha=.55, label=lbl, color=col, density=True)
    ax.set(xlabel=c, ylabel="density", title=c)
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(RESULTS / "capture_geometry_domain_shift.png", dpi=140, bbox_inches="tight")
plt.show()

dom = (geo.assign(domain=np.where(geo.disaster.isin(fire_events), "wildfire (OOD)", "non-fire"))
          .groupby("domain")[meta_cols].agg(["mean", "std"]).round(2))
display(dom)
dom.to_csv(RESULTS / "capture_geometry_by_domain.csv")

**What we see (reference run).** Capture geometry is not identical between the two domains:

| | GSD | off-nadir | sun elevation |
|---|---|---|---|
| non-fire | 2.21 ± 0.52 | 28.5 ± 9.9 | 64.4 ± 7.5 |
| wildfire | 2.12 ± 0.31 | 25.6 ± 6.0 | **55.0 ± 15.9** |

GSD and off-nadir angle are close. **Sun elevation is not**: the wildfire mean is 9° lower with more
than twice the spread, and the histogram shows a fire mode near 35° that has no counterpart in
training. Low sun means long shadows and a different look for the same structure.

**Why it matters.** Part of any transfer gap could be acquisition geometry rather than fire damage
looking different. Claiming "damage does not transfer across disaster types" when the real driver is
shadow length would be unsupported.

**The honest caveat.** The histograms are spiky because every scene in an event shares one
acquisition. Statistically this is **19 points, not 11,034**, and the geometry is fully confounded
with event identity — "wildfire" and "the captures that happen to be wildfire" cannot be separated
on this data. That belongs in the limitations section rather than being quietly ignored.

**Downstream.** Flip, 90° rotation and scale augmentation become an evidence-backed robustness
measure rather than a reflex.

*(Note: the `gsd` field reads ~2.2, while the xBD paper quotes sub-metre imagery — the field most
likely records pre-pansharpening GSD. Do not quote both figures side by side without checking. The
comparison between domains is still valid; only the absolute unit is uncertain.)*

### 3.6 Image properties — dimensions, channels, and readability

Verified on a random sample of scenes rather than all 22,068 files: decoding every 1024×1024 PNG
would take longer than the rest of the notebook combined, and a random sample is enough to confirm a
uniform format (and to *detect* it if the format is not uniform).

In [ ]:
set_seed()
per_split = max(QC_SAMPLE_SCENES // scenes.split.nunique(), 1)
qc_scenes = pd.concat([g.sample(min(len(g), per_split), random_state=SEED)
                       for _, g in scenes.groupby("split")], ignore_index=True)
print(f"QC sample: {len(qc_scenes)} scenes ({2 * len(qc_scenes)} image files)")


def probe_image(path):
    """Open header only where possible; return format facts plus any error."""
    try:
        with Image.open(path) as im:
            w, h, mode, fmt = im.width, im.height, im.mode, im.format
        return {"path": path, "w": w, "h": h, "mode": mode, "format": fmt, "error": None}
    except Exception as e:
        return {"path": path, "w": None, "h": None, "mode": None, "format": None,
                "error": f"{type(e).__name__}: {e}"}


with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    probes = pd.DataFrame(list(ex.map(probe_image,
                                      qc_scenes.img_pre.tolist() + qc_scenes.img_post.tolist())))

print("\nresolution x mode x format:")
fmt_table = probes.groupby(["w", "h", "mode", "format"], dropna=False).size().rename("files").to_frame()
display(fmt_table)
fmt_table.to_csv(RESULTS / "image_quality_sample.csv")
print("header-level read errors:", int(probes.error.notna().sum()))
if probes.error.notna().any():
    display(probes[probes.error.notna()].head(10))

**What we see (reference run).** All 800 sampled files are 1024×1024 RGB PNG, 8-bit, with zero read
errors.

**Why it matters.** Two items the project brief lists as cleaning work are already done by the
dataset: there is no resolution standardisation and no band alignment to perform. More importantly,
**pre and post tiles are already co-registered** by the xBD authors — pixel (x, y) is the same ground
location in both — which is what makes cropping the *same* box from both images valid. Without that
the entire siamese design would be comparing different places.

**Downstream.** Preprocessing reduces to: crop by bbox, resize, scale to [0,1], ImageNet-normalise.
No warping, no resampling.

### 3.7 Representative samples — what the four classes actually look like

In [ ]:
def load_patch(img_path, box, pad=BBOX_PAD, size=PATCH_SIZE, full=None):
    """Crop one padded, resized RGB patch. `full` lets a caller reuse an already-open image."""
    im = full if full is not None else Image.open(img_path).convert("RGB")
    minx, miny, maxx, maxy = box
    W, H = im.size
    l, t = max(0, int(minx) - pad), max(0, int(miny) - pad)
    r, b = min(W, int(maxx) + pad), min(H, int(maxy) + pad)
    if r - l < 2 or b - t < 2:
        return None
    return np.asarray(im.crop((l, t, r, b)).resize((size, size), Image.BILINEAR), np.uint8)


def show_examples(pool, title, fname, n_per_class=5, size=96, scenes_per_class=3):
    """Grid: rows = damage class, columns = examples, each cell = pre | post side by side.

    I/O is the bottleneck, not plotting. Drawing 5 independent random buildings per class would
    touch 10 different 1024x1024 tiles per row. Instead we pick a few scenes that are RICH in that
    class (one per event, to keep the columns visually diverse), decode each tile once and reuse it
    for every example it contributes.
    """
    cache = {}

    def tile(path):
        if path not in cache:
            cache[path] = Image.open(path).convert("RGB")
        return cache[path]

    fig, axes = plt.subplots(NUM_CLASSES, n_per_class,
                             figsize=(2.1 * n_per_class, 2.3 * NUM_CLASSES), squeeze=False)
    for c in range(NUM_CLASSES):
        for j in range(n_per_class):
            axes[c, j].axis("off"); axes[c, j].grid(False)

        cand = pool[pool.label == c]
        if cand.empty:
            continue
        # richest scene per event, so one decode serves several columns
        ranked = (cand.groupby(["scene_id", "disaster"]).size().rename("n").reset_index()
                      .sort_values("n", ascending=False)
                      .drop_duplicates("disaster").head(scenes_per_class))
        sub = cand[cand.scene_id.isin(ranked.scene_id)]
        picks = sub.sample(min(n_per_class, len(sub)), random_state=SEED)

        for j, (_, r) in enumerate(picks.iterrows()):
            box = (r.minx, r.miny, r.maxx, r.maxy)
            pre  = load_patch(None, box, size=size, full=tile(r.img_pre))
            post = load_patch(None, box, size=size, full=tile(r.img_post))
            if pre is None or post is None:
                continue
            axes[c, j].imshow(np.concatenate(
                [pre, np.full((size, 3, 3), 255, np.uint8), post], axis=1))
            axes[c, j].set_title(r.disaster, fontsize=6)

        if len(picks):
            axes[c, 0].axis("on"); axes[c, 0].set_xticks([]); axes[c, 0].set_yticks([])
            axes[c, 0].set_ylabel(DAMAGE_CLASSES[c], fontsize=8)

    fig.suptitle(f"{title}\n(each cell: PRE | POST)", fontsize=11)
    plt.tight_layout()
    fig.savefig(RESULTS / fname, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"decoded {len(cache)} tiles for this figure")


sample_pool = (buildings[buildings.label.notna()]
               .merge(scenes[["scene_id", "img_pre", "img_post"]], on="scene_id"))
set_seed()
if SHOW_EXAMPLE_GRIDS:
    show_examples(sample_pool[sample_pool.split == "train"], "TRAIN domain (non-fire)",
                  "training_examples_by_class.png")
else:
    print("skipped (SHOW_EXAMPLE_GRIDS = False)")

In [ ]:
ood_pool = sample_pool[sample_pool.split == "test_ood"]
if SHOW_EXAMPLE_GRIDS and len(ood_pool):
    show_examples(ood_pool, "HELD-OUT domain (wildfire)", "ood_examples_by_class.png")
elif not len(ood_pool):
    print("no test_ood scenes on disk - extract tier3.tar (or hold) to see this figure")
else:
    print("skipped (SHOW_EXAMPLE_GRIDS = False)")

**What we see.** The two domains are visibly different problems. Non-fire post images usually retain
the structure with altered geometry — torn roofs, debris, standing water. Wildfire damage collapses
buildings to a uniform grey scar with the footprint faintly visible, while undamaged buildings often
sit in surroundings that changed completely (scorched vegetation) with the roof untouched.

**The prediction this generates.** A change-detection model learns "the scene changed" ⇒ "the
building is damaged". Under wildfire the *context* changes for undamaged buildings too, so we should
expect **false positives on `no-damage`** in `test_ood`.

**Downstream.** This is falsifiable, and the evaluation notebook tests it directly.
→ **Confirmed on the reference run: 37.2% of truly undamaged wildfire buildings were flagged as
damaged.** It also justifies keeping `BBOX_PAD` modest — more surrounding context makes the model
more sensitive to exactly the signal that misleads it here.

## 4 · Data cleaning and quality diagnostics

**Operating principle:** nothing is removed silently. Every rule below states *what* it removes,
*why*, *at what threshold*, and *how many* samples it costs. Rules whose thresholds cannot be
justified from the data produce a **flag and a gallery for manual review** instead of a deletion.

We accumulate decisions in a ledger and print it at the end of the section.

In [ ]:
CLEANING_LEDGER = []

def log_rule(rule, scope, removed, kept, why, threshold=""):
    CLEANING_LEDGER.append({"rule": rule, "scope": scope, "removed": int(removed),
                            "kept": int(kept), "threshold": str(threshold), "rationale": why})
    pct = 100 * removed / max(removed + kept, 1)
    print(f"[{rule}] removed {removed:,} ({pct:.2f}%), kept {kept:,}  | {threshold}")

### 4.1 Rule — missing or unpaired files

A scene is usable only if **pre image, post image and post label** all exist. A missing pre image
makes change detection impossible; a missing post label leaves the sample unlabelled.

This rule was already *applied* in the coverage cell (2.2) so the label scan would not fail on
absent files; here it is *recorded*, against the full `splits.csv` index, so the ledger states the
true cost on this machine.

In [ ]:
print("per-file presence, measured against the full splits.csv index:")
pres = pd.DataFrame({c: scenes_all[c].map(os.path.exists) for c in NEEDED})
display(pres.sum().rename("present").to_frame().assign(missing=len(scenes_all) - pres.sum()))

log_rule("4.1 missing/unpaired files", "scenes",
         removed=(~scenes_all.files_ok).sum(), kept=scenes_all.files_ok.sum(),
         why="change detection needs pre+post+label; an incomplete scene cannot be used",
         threshold="all three files must exist")

if (~scenes_all.files_ok).any():
    print("\nmissing scenes by origin x split (these are absent archives, not corrupt data):")
    display(pd.crosstab(scenes_all.loc[~scenes_all.files_ok, "origin"],
                        scenes_all.loc[~scenes_all.files_ok, "split"]))

### 4.2 Rule — corrupted / unreadable files

Two distinct failure modes, checked separately:
- **label JSON** unparseable (already captured during the scan in 2.3)
- **image PNG** truncated or with a broken CRC — a header probe is not enough, so we fully decode
  the QC sample. `LOAD_TRUNCATED_IMAGES` was set to `False` at import precisely so these raise.

In [ ]:
def verify_decode(path):
    """Fully decode an image; returns None on success, else the error string."""
    try:
        with Image.open(path) as im:
            im.load()
        return None
    except Exception as e:
        return f"{type(e).__name__}: {e}"


decode_files = qc_scenes.img_pre.tolist() + qc_scenes.img_post.tolist()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    decode_errs = list(ex.map(verify_decode, decode_files))

n_bad_img = sum(e is not None for e in decode_errs)
print(f"fully decoded {len(decode_files)} images from the QC sample")
for f, e in zip(decode_files, decode_errs):
    if e:
        print("  CORRUPT:", Path(f).name, "->", e)

log_rule("4.2a corrupted images", f"QC sample of {len(decode_files)} images",
         removed=n_bad_img, kept=len(decode_files) - n_bad_img,
         why="a truncated PNG raises mid-training and kills the run",
         threshold="PIL full decode must succeed")
log_rule("4.2b unreadable labels", "all scanned scenes",
         removed=scene_meta.label_error.notna().sum(),
         kept=scene_meta.label_error.isna().sum(),
         why="an unparseable JSON yields no labels for the whole scene",
         threshold="json.loads must succeed")

print("\nNote: full decoding is done on a sample only. The patch-extraction loop in the model "
      "notebooks wraps every read in try/except and reports any file that fails there, so the "
      "remaining scenes are covered without a second full pass over 22,068 PNGs.")

### 4.3 Diagnostic — smoke / cloud occlusion

This is the team's open question: *"we need to remove images that are totally covered by smoke."*

**Finding: xBD has no occlusion label.** There is no cloud, smoke or quality field anywhere in the
label JSON — the `metadata` block contains only capture geometry (verified in 2.3), and `splits.csv`
carries no such column either. So an automatic rule cannot be grounded in ground truth.

**The closest thing to a real label is `subtype = "un-classified"`.** Annotators used it when a
building could not be assessed, and occlusion is one of the main reasons that happens. It is a
*proxy*, not an occlusion label — it also covers ambiguous or partially-visible structures — but it
is a human judgement recorded in the ground truth, which is strictly better than a pixel threshold
we invented. We rank scenes by it below and then check the ranking against the pixels.

In [ ]:
unc_stats = scene_meta[scene_meta.n_buildings >= 10].copy()
unc_stats["unc_frac"] = unc_stats.n_unclassified / unc_stats.n_buildings

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for sp in SPLITS:
    v = unc_stats.loc[unc_stats.split == sp, "unc_frac"]
    if len(v):
        axes[0].hist(v, bins=np.linspace(0, 1, 41), histtype="step", lw=1.6, label=sp)
axes[0].set(xlabel="fraction of polygons marked un-classified", ylabel="scenes",
            yscale="log", title="Un-classified rate per scene (>=10 buildings)")
axes[0].legend(fontsize=8)

by_ev = unc_stats.groupby("disaster").unc_frac.mean().mul(100).sort_values()
by_ev.plot(kind="barh", ax=axes[1],
           color=["crimson" if d in fire_events else "steelblue" for d in by_ev.index])
axes[1].set(xlabel="mean un-classified rate (%)", ylabel="",
            title="Un-classified rate per event (red = wildfire)")
axes[1].tick_params(labelsize=8)
plt.tight_layout()
fig.savefig(RESULTS / "unclassified_rate_by_event.png", dpi=140, bbox_inches="tight")
plt.show()

for thr in (0.5, 0.8, 0.95):
    n = int((unc_stats.unc_frac >= thr).sum())
    print(f"scenes with >= {thr:.0%} un-classified: {n:5d}  "
          f"({100 * n / max(len(unc_stats), 1):.2f}% of scenes with >=10 buildings)")

### 4.3b — do the pixels agree with the annotators?

If `un-classified` really tracks occlusion, then high-rate scenes should also look occluded to
simple image statistics. Smoke and thin cloud on 3-band optical imagery are **bright, low-contrast,
low-saturation, and edge-poor**. We compute those four statistics on the post-disaster tile for the
QC sample plus every high-`un-classified` scene, and test whether the two signals correlate.

If they do not correlate, the honest conclusion is that neither signal is trustworthy alone — and we
say so rather than shipping a filter.

In [ ]:
def occlusion_stats(path, side=256):
    """Cheap whole-tile occlusion statistics on a downscaled copy of the post image."""
    try:
        with Image.open(path) as im:
            a = np.asarray(im.convert("RGB").resize((side, side), Image.BILINEAR), np.float32) / 255.
    except Exception:
        return dict(brightness=np.nan, contrast=np.nan, saturation=np.nan, edges=np.nan)
    g = a.mean(axis=2)
    gy, gx = np.gradient(g)
    return dict(
        brightness=float(g.mean()),                                   # smoke/cloud raises this
        contrast=float(g.std()),                                      # and flattens this
        saturation=float((a.max(2) - a.min(2)).mean()),               # smoke is near-grey
        edges=float(np.hypot(gx, gy).mean()),                         # texture disappears
    )


unc_top = unc_stats.nlargest(300, "unc_frac").scene_id
qc_ids  = pd.Index(qc_scenes.scene_id).union(unc_top)
occ_src = scenes[scenes.scene_id.isin(qc_ids)].copy()

OCC_CACHE = CACHE / f"occlusion__{COVERAGE_TAG}.parquet"
if OCC_CACHE.exists():
    occ = pd.read_parquet(OCC_CACHE)
    print("loaded occlusion stats from cache")
else:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
        stats = list(ex.map(occlusion_stats, occ_src.img_post.tolist()))
    occ = pd.concat([occ_src[["scene_id", "disaster", "split", "img_pre", "img_post"]]
                     .reset_index(drop=True), pd.DataFrame(stats)], axis=1)
    occ.to_parquet(OCC_CACHE, index=False)
    print(f"computed occlusion stats for {len(occ)} tiles in {time.time() - t0:.0f}s")

occ = occ.merge(unc_stats[["scene_id", "unc_frac", "n_buildings"]], on="scene_id", how="left")
occ_c = occ.dropna(subset=["unc_frac", "brightness"])

print(f"\ncorrelation of un-classified rate with image statistics (n={len(occ_c)}):")
rho = (occ_c[["unc_frac", "brightness", "contrast", "saturation", "edges"]]
       .corr(method="spearman")["unc_frac"].drop("unc_frac").round(3)
       .rename("spearman rho").to_frame())
display(rho)
rho.to_csv(RESULTS / "occlusion_proxy_correlation.csv")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, c in zip(axes, ["brightness", "contrast", "saturation", "edges"]):
    ax.scatter(occ_c[c], occ_c.unc_frac, s=6, alpha=.35)
    ax.set(xlabel=c, ylabel="un-classified rate" if c == "brightness" else "")
plt.suptitle("Annotator 'cannot assess' rate vs. image statistics", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS / "occlusion_proxy_correlation.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------- composite candidate score
# Ranking only. It is NOT used to delete anything; it selects what a human looks at.
z = lambda s: (s - s.mean()) / s.std(ddof=0)
occ["occ_score"] = (z(occ.brightness) - z(occ.contrast) - z(occ.saturation) - z(occ.edges)) / 4
cands = occ.dropna(subset=["occ_score"]).nlargest(OCCLUSION_TOP_N, "occ_score")

print("Top occlusion candidates for MANUAL review:")
display(cands[["scene_id", "disaster", "split", "unc_frac", "brightness",
               "contrast", "saturation", "edges", "occ_score"]].round(3).reset_index(drop=True))

n = max(min(OCCLUSION_TOP_N, len(cands)), 1)
fig, axes = plt.subplots(2, n, figsize=(1.9 * n, 4.4), squeeze=False)
for j in range(n):
    if j >= len(cands):
        axes[0, j].axis("off"); axes[1, j].axis("off")
        continue
    r = cands.iloc[j]
    for i, p in enumerate([r.img_pre, r.img_post]):
        ax = axes[i, j]; ax.axis("off"); ax.grid(False)
        try:
            with Image.open(p) as im:
                ax.imshow(im.convert("RGB").resize((160, 160)))
        except Exception:
            pass
    axes[0, j].set_title(f"{r.disaster}\nunc={r.unc_frac:.2f}" if pd.notna(r.unc_frac)
                         else r.disaster, fontsize=6)
axes[0, 0].set_ylabel("PRE"); axes[1, 0].set_ylabel("POST")
plt.suptitle("Occlusion candidates - top row PRE, bottom row POST (manual review)", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS / "occlusion_gallery.png", dpi=140, bbox_inches="tight")
plt.show()

**What we measured (reference run).**

| statistic | Spearman ρ with un-classified rate |
|---|---|
| edge density | **−0.366** |
| brightness | +0.116 |
| saturation | −0.095 |
| contrast | −0.035 |

Only **edge density** correlates meaningfully. Brightness is weak and contrast is essentially zero,
so the composite score is really measuring "texture-poor", not "bright and flat" as expected.

**What the gallery shows.** The top candidates are **not wildfire**. They are `sunda-tsunami` and
`lower-puna-volcano` tiles under thick cloud — the worst has brightness 0.99 with contrast 0.049,
which is a tile that is almost pure white. Flagged at the 98th percentile: 11 scenes in `train`,
3 in `val`, **0 in `test_id`, 0 in `test_ood`**.

**This reverses the concern we started with.** The team's worry was smoke hiding wildfire damage.
The measurement says the real occlusion in xBD is *cloud over tsunami and volcano scenes*, and it
sits in the training pool, not in either test set.

**Decision: nothing is removed.** Reasons, in order of weight:

1. **No ground truth to validate a threshold.** Any cut point would be tuned by eye on a handful of
   images. A rule that deletes data and cannot be validated is an unmeasurable bias.
2. **There is almost nothing to gain.** 14 flagged scenes out of 683 inspected, none in a test split.
   The effect on any reported number is negligible.
3. **Occlusion is part of the deployment problem.** A system running on real post-disaster imagery
   meets cloud and smoke. Deleting it measures a scenario that does not exist.
4. **A safer patch-level filter already exists** — patch extraction rejects crops that are
   essentially pure white or pure black, applied identically to every split.

Candidates are exported to `occlusion_candidates.csv`. If the team wants removal, the defensible
route is to hand-label the top ~200, measure the score's false-positive rate, and report the result
both with and without the filter.

In [ ]:
FLAG_THRESHOLD = occ.occ_score.quantile(0.98)      # flag only, no deletion
occ["occlusion_flag"] = occ.occ_score >= FLAG_THRESHOLD
occ[["scene_id", "disaster", "split", "unc_frac", "occ_score", "occlusion_flag"]] \
    .to_csv(RESULTS / "occlusion_candidates.csv", index=False)

flag_by_split = occ.groupby("split").occlusion_flag.agg(["sum", "size"])
flag_by_split.columns = ["flagged", "inspected"]
display(flag_by_split)

log_rule("4.3 smoke/cloud occlusion", "scenes", removed=0, kept=len(occ),
         why="no occlusion ground truth exists; removal would bias test_ood, the split under study",
         threshold=f"FLAGGED ONLY at occ_score >= p98 ({FLAG_THRESHOLD:.3f}) -> occlusion_candidates.csv")

### 4.4 Rule — invalid and unusable labels

In [ ]:
n_unc_total = int((buildings.subtype == UNCLASSIFIED).sum())
n_valid     = int(buildings.label.notna().sum())

log_rule("4.4a un-classified subtype", "building polygons",
         removed=n_unc_total, kept=n_valid,
         why="not a damage level - the annotator could not assess the building; keeping it would "
             "add a 5th class that has no operational meaning and pollutes all four real ones",
         threshold=f"subtype == '{UNCLASSIFIED}'")

bad_geo = buildings[(buildings.bw <= 0) | (buildings.bh <= 0)]
log_rule("4.4b degenerate polygons", "building polygons",
         removed=len(bad_geo), kept=len(buildings) - len(bad_geo),
         why="zero-area bbox cannot be cropped", threshold="bbox width or height <= 0")

print("\nun-classified share per split (this is the cost of rule 4.4a, and it is NOT uniform):")
unc_split = (buildings.assign(unc=buildings.subtype.eq(UNCLASSIFIED))
             .groupby("split").unc.agg(["sum", "size"]))
unc_split["pct"] = (100 * unc_split["sum"] / unc_split["size"]).round(2)
unc_split.columns = ["un-classified", "polygons", "%"]
unc_split = unc_split.reindex(SPLITS)
display(unc_split)
unc_split.to_csv(RESULTS / "unclassified_rate_by_split.csv")

**What we see (reference run).** The `un-classified` rate is 3.67% in train, 4.08% in val, 2.09% in
`test_id` and 3.30% in `test_ood`.

**Why it matters.** Dropping `un-classified` is unavoidable — there is no damage level to train
against, and the official xView2 metric ignores it too. The risk was that the rate would be much
higher in `test_ood`, which would mean discarding more hard wildfire cases than hard hurricane cases
and flattering the transfer result.

**It did not happen.** `test_ood` (3.30%) sits below `train` (3.67%), so this rule does not tilt the
comparison. The remaining honest caveat is smaller but real: every reported number is measured on
buildings that human annotators *were able to assess*, so true zero-shot difficulty is slightly
higher than what we report.

### 4.5 Rule — degenerate footprint size

In [ ]:
valid = buildings[buildings.label.notna()].copy()
valid = valid[(valid.bw > 0) & (valid.bh > 0)]
too_small = (valid.bw < MIN_BBOX_PX) | (valid.bh < MIN_BBOX_PX)

log_rule("4.5 tiny footprints", "labelled building polygons",
         removed=too_small.sum(), kept=(~too_small).sum(),
         why=f"after {BBOX_PAD}px padding and upscaling to {PATCH_SIZE}px these are interpolation "
             "artefacts, not observations of a building; also a likely source of label noise",
         threshold=f"bbox side < {MIN_BBOX_PX}px")

print("\nremoval rate by split (must be roughly uniform, or the rule biases the comparison):")
display((100 * too_small.groupby(valid.split).mean()).round(2).reindex(SPLITS)
        .rename("% removed").to_frame())

valid = valid[~too_small].copy()

### 4.6 Rule — duplicates and near-duplicates

Exact duplicates are impossible by construction: `prepare_splits.py` already proved every filename is
unique across the four origin directories, and `scene_id` is unique after the pivot. The real risk is
**near-duplicates** — overlapping or adjacent tiles of the same area — because those could put nearly
the same building in `train` and in an evaluation split.

We test with a perceptual difference hash on the post tile. `dhash` is implemented directly in numpy
rather than pulling in `imagehash`, to keep the notebook dependency-free.

In [ ]:
def dhash_bits(path, hash_size=8):
    """64-bit perceptual difference hash, returned as a '0'/'1' string.

    Kept as a bit string rather than a 64-bit integer on purpose: a full 64-bit hash overflows
    numpy's signed int64, and the string form survives the parquet round-trip unchanged.
    """
    try:
        with Image.open(path) as im:
            a = np.asarray(im.convert("L").resize((hash_size + 1, hash_size), Image.BILINEAR), np.int16)
    except Exception:
        return None
    return "".join("1" if b else "0" for b in (a[:, 1:] > a[:, :-1]).flatten())


HASH_CACHE = CACHE / f"hashes__{COVERAGE_TAG}.parquet"
if HASH_CACHE.exists():
    hashes = pd.read_parquet(HASH_CACHE)
    print("loaded hashes from cache")
else:
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
        hs = list(ex.map(dhash_bits, qc_scenes.img_post.tolist()))
    hashes = qc_scenes[["scene_id", "disaster", "split"]].copy()
    hashes["h"] = hs
    hashes = hashes.dropna(subset=["h"]).reset_index(drop=True)
    hashes.to_parquet(HASH_CACHE, index=False)
    print(f"hashed {len(hashes)} post tiles")

# pairwise Hamming distance over the QC sample (a few hundred tiles -> a trivially small matrix)
B = np.array([[c == "1" for c in s] for s in hashes.h], dtype=bool)      # n x 64
ham = (B[:, None, :] != B[None, :, :]).sum(axis=-1).astype(np.int16)
np.fill_diagonal(ham, 64)

NEAR_DUP_BITS = 6          # <=6/64 differing bits: visually near-identical tiles
i, j = np.where(np.triu(ham <= NEAR_DUP_BITS, k=1))
pairs = pd.DataFrame({
    "a": hashes.scene_id.values[i], "b": hashes.scene_id.values[j],
    "split_a": hashes.split.values[i], "split_b": hashes.split.values[j],
    "bits": ham[i, j],
})
cross = pairs[pairs.split_a != pairs.split_b]

print(f"\nnear-duplicate pairs within the QC sample (<= {NEAR_DUP_BITS} bits): {len(pairs)}")
print(f"of which CROSS-SPLIT (the ones that would leak):       {len(cross)}")
if len(cross):
    display(cross.head(20))
pairs.to_csv(RESULTS / "near_duplicate_pairs_sample.csv", index=False)

log_rule("4.6 near-duplicates", f"QC sample of {len(hashes)} scenes",
         removed=0, kept=len(hashes),
         why="flagged for inspection; xBD tiles are non-overlapping by construction and the split "
             "is scene-level, so cross-split leakage is not expected",
         threshold=f"dhash Hamming distance <= {NEAR_DUP_BITS}/64")

**What the check returned, and what it means (reference run).** 28 near-duplicate pairs in the
400-scene QC sample, 19 of them cross-split. That looks alarming, so it was investigated rather than
waved through.

**They are not duplicates.** Every cross-split pair links *different events on different continents*
— `portugal-wildfire` matched against `nepal-flooding`, `hurricane-harvey`, `sunda-tsunami`,
`palu-tsunami` and `moore-tornado` simultaneously. One tile cannot be the same location as five
unrelated disasters. What is actually happening is dhash degeneracy: a 64-bit difference hash of a
low-texture tile (open water, unbroken forest, solid cloud) collapses toward a constant, so all flat
tiles collide with each other regardless of content.

**Conclusion: no evidence of leakage.** This is consistent with the design — xBD tiles a disaster
area into non-overlapping chips, and the split is assigned at **scene** level, so pre and post can
never separate and no building can appear in two splits.

**Stated limit.** The test covers 400 scenes, not all 11,034, so it is a spot check. A content-aware
duplicate test (embedding distance rather than a perceptual hash) would be the right upgrade if the
team wants a stronger guarantee.

### 4.7 Cleaning ledger

In [ ]:
ledger = pd.DataFrame(CLEANING_LEDGER)
display(ledger)
ledger.to_csv(RESULTS / "cleaning_ledger.csv", index=False)

print(f"\nlabelled, usable building polygons after cleaning: {len(valid):,} "
      f"({100 * len(valid) / max(len(buildings), 1):.1f}% of all polygons)")
display(valid.groupby("split").size().reindex(SPLITS).rename("usable polygons").to_frame())

# Hand-off to 02_data_split.ipynb. Coverage-tagged for the same reason the scan caches are.
CLEAN_PARQUET = CACHE / f"clean_buildings__{COVERAGE_TAG}.parquet"
valid.to_parquet(CLEAN_PARQUET, index=False)
print("\nwrote", CLEAN_PARQUET)
print("wrote figures + tables to", RESULTS)

## 5 · What this section established

**Cleaning cost on the reference run: 6.7% of polygons** — 3.29% un-classified plus 3.51% footprints
under 8 px — at a removal rate that is uniform across splits (2.99–3.77%), so it cannot tilt the
ID→OOD comparison.

**Findings that shaped the pipeline:**

- **Wildfire damage is bimodal.** Non-fire is 70.2 / 9.9 / 7.2 / 6.5 % across the four classes,
  wildfire is 78.8 / **1.2** / **1.1** / **14.8** %. Fire leaves buildings intact or destroys them;
  intermediate states barely exist.
- **54.1% of wildfire scenes contain no buildings at all** (median 0 per scene), against 9.8% in
  train — which is why scenes are sampled subject to a minimum building count.
- **No occlusion label exists in xBD.** The worst-occluded tiles are cloud over `sunda-tsunami` and
  `lower-puna-volcano`, **not** smoke over wildfire — 14 flagged scenes, all in train/val, none in
  either test set. Nothing was removed.
- **`PATCH_SIZE = 64`, `BBOX_PAD = 10`** follow directly from the measured footprint distribution.
- **Sun elevation differs by 9° between domains** and is fully confounded with event identity —
  a stated limitation, not a fixable one.

**Next:** [`02_data_split.ipynb`](02_data_split.ipynb) — split integrity, leakage checks, and the
event-stratified scene sampling that feeds patch extraction.